# Projeto Fictus | Análise Logística — Bloco 2: Viabilidade Econômica da Internalização

---

## Pergunta Central do Bloco
> **A internalização melhora ou deteriora a margem consolidada — e a partir de qual volume?**

---

## Contexto do Bloco

Com o diagnóstico do modelo atual estabelecido, a pergunta muda de descrição para decisão: quanto custaria resolver os gargalos identificados através da internalização?

Analisamos o ponto de equilíbrio (Break-even) onde o custo fixo de uma operação própria é compensado pelo volume de vendas. Esta análise é estruturada de forma universal, permitindo decidir se vale a pena assumir a complexidade operacional em troca de maior controle e margem.

O coração deste bloco é o make vs buy (Fazer ou Comprar) econômico. A comparação não é simples porque os três modelos têm estruturas de custo diferentes:  
- o modelo terceirizado tem custo quase totalmente variável (o cliente paga o frete); 
- o modelo próprio tem custo fixo alto mais um componente variável menor, porém só compensa a partir de um volume mínimo — o break-even de pedidos por mês.
- e o "Modelo Híbrido" que é como "ter uma frota própria para as cidades onde há mais vendas e continuar com terceiros para o resto do país, equilibrando risco e investimento.

**Este bloco investiga:**
1. Qual seria o custo fixo de uma operação internalizada?
2. Existe um volume a partir do qual a internalização se paga — e já estamos nele?
3. Qual é o custo de capital e o payback do investimento?
4. Um modelo híbrido captura quanto do benefício com qual fração do investimento?
5. Se a internalização não for viável no volume atual, em quanto tempo o crescimento orgânico do negócio atingiria o break-even — e esse horizonte é aceitável para o comprador?

---

## Nota sobre Premissas
Todas as premissas do modelo estão declaradas na célula de carregamento e são auditáveis individualmente. Altere qualquer valor e toda a análise se atualiza automaticamente.

---


## Configuração do Ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path

try:
    _base = Path(__file__).resolve().parent
except NameError:
    _base = Path().resolve()
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(_base)
DIR_LOG  = BASE_DIR / "data" / "logistics"
DIR_EXPORTS = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

COR_FRETE   = "#C0392B"; COR_RECEITA = "#1B4F72"; COR_MARGEM  = "#27AE60"
COR_ALERTA  = "#E74C3C"; COR_NEUTRO  = "#7F8C8D"; COR_DESTAQUE= "#E67E22"
COR_ROXO    = "#8E44AD"; COR_VERDE   = "#1E8449"

sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({"figure.dpi":150,"savefig.dpi":150,"savefig.bbox":"tight",
    "font.family":"sans-serif","axes.spines.top":False,"axes.spines.right":False})
def fmt_brl(x,pos=None):
    if abs(x)>=1_000_000: return f"R$ {x/1_000_000:.1f}M"
    elif abs(x)>=1_000: return f"R$ {x/1_000:.0f}K"
    return f"R$ {x:.2f}"
def fmt_pct(x,pos=None): return f"{x:.1f}%"
def salvar(fig,nome):
    caminho = DIR_EXPORTS / f"{nome}.png"
    fig.savefig(caminho); print(f"  -> Salvo: {caminho.name}")
print("Ambiente configurado.")


## Carregamento e Premissas

In [ ]:
def ler(f,**kw):
    df=pd.read_csv(DIR_LOG/f,low_memory=False,**kw); df.columns=df.columns.str.strip(); return df
log_fato   = ler("log_fato.csv")
log_mensal = ler("log_mensal.csv")
log_trim   = ler("log_trimestral.csv")
log_rota   = ler("log_rota.csv")
log_cat    = ler("log_categoria.csv")
for col in ["data_compra","data_entrega_cliente"]:
    if col in log_fato.columns: log_fato[col] = pd.to_datetime(log_fato[col],errors="coerce")
for col in ["preco","valor_frete","lead_time_dias","atraso_dias","nota_review"]:
    if col in log_fato.columns: log_fato[col] = pd.to_numeric(log_fato[col],errors="coerce")
periodos_ord = sorted(log_fato["periodo"].dropna().unique())
print(f"Dados carregados: {len(log_fato):,} pedidos | {periodos_ord[0]} a {periodos_ord[-1]}")

# ─── Premissas declaradas do modelo de internalização ─────────────────────────
# Todas auditáveis — altere qualquer valor e a análise inteira se atualiza
CUSTO_FIXO_MENSAL   = 180_000   # R$/mês: frota + galpão + RH (benchmark mercado BR)
CUSTO_VAR_POR_PED   = 12.50     # R$/pedido: custo variável operação própria
REDUCAO_FRETE_CLI   = 0.20      # -20% no frete cobrado ao cliente (repasse conservador)
AUMENTO_VOLUME      = 0.10      # +10% de pedidos por menor frete ao cliente
MESES_MATURACAO     = 6         # meses até operação atingir eficiência plena
CUSTO_IMPLANTACAO   = 500_000   # R$: capital inicial (frota, infra, tecnologia)
CUSTO_REVERSAO_EST  = 300_000   # R$: estimativa de custo para desfazer operação própria
print("\nPremissas do modelo de internalização:")
print(f"  Custo fixo mensal    : R$ {CUSTO_FIXO_MENSAL:,.0f}")
print(f"  Custo variável/pedido: R$ {CUSTO_VAR_POR_PED:.2f}")
print(f"  Reducao frete cliente: {REDUCAO_FRETE_CLI*100:.0f}%")
print(f"  Aumento de volume    : +{AUMENTO_VOLUME*100:.0f}%")
print(f"  Capital de implant.  : R$ {CUSTO_IMPLANTACAO:,.0f}")


---

## Análise 1 — Qual seria o custo fixo de uma operação internalizada?

> *"O custo de uma operação que ainda não existe é sempre estimado. O modelo trabalha com premissas declaradas e benchmarks de mercado — da mesma forma que qualquer estudo de viabilidade real é conduzido. Frota, galpão, mão de obra e tecnologia são decompostos como componentes separados, para que cada premissa possa ser auditada e ajustada individualmente."*

**Framework:** Make vs Buy Analysis + TCO
**Entrega:** Modelagem detalhada do custo fixo mensal com premissas declaradas e auditáveis

**Como este script responde à pergunta:**
> O script constrói a estrutura de custos da operação internalizada em dois níveis. O custo fixo é independente do volume — frota, galpão, pessoal. O custo variável escala com o número de pedidos. A comparação dos dois modelos em uma curva de custo total por volume mostra exatamente onde cada um é superior.
>
> 1. **Decomposição do custo fixo:** Gráfico de pizza mostrando a composição estimada do custo fixo mensal por componente (frota, galpão, RH, tecnologia). Cada componente tem sua premissa declarada ao lado.
> 2. **Curva de custo total por modelo:** Plota o custo total mensal de cada modelo em função do volume de pedidos. O ponto de cruzamento das duas curvas é o break-even — a partir daí, o modelo próprio é mais barato por pedido.

**Análise do Resultado:**
Esta análise quantifica a nova estrutura de gastos que a empresa passaria a ter. Diferente do modelo atual, onde o custo só existe se houver venda, o custo fixo representa o compromisso financeiro mensal independente do volume. Compreender este valor é vital para o comprador avaliar o risco de alavancagem operacional do ativo.

In [ ]:
# Decomposição estimada do custo fixo
componentes = {
    "Frota (depreciacao + manut.)": CUSTO_FIXO_MENSAL * 0.40,
    "Galpao (aluguel + ops)":       CUSTO_FIXO_MENSAL * 0.25,
    "Recursos Humanos":             CUSTO_FIXO_MENSAL * 0.25,
    "Tecnologia e sistemas":        CUSTO_FIXO_MENSAL * 0.10,
}
n_pedidos_atual = log_fato["id_pedido"].nunique()
n_pedidos_mensal = log_mensal["n_pedidos"].mean()

# Curva de custo por volume
volumes = np.arange(1000, n_pedidos_mensal * 2.5, 100)
custo_terceirizado = log_fato["valor_frete"].mean() * volumes   # frete medio x volume
custo_proprio      = CUSTO_FIXO_MENSAL + CUSTO_VAR_POR_PED * volumes
custo_hibrido      = CUSTO_FIXO_MENSAL * 0.5 + CUSTO_VAR_POR_PED * 0.6 * volumes  # modelo parcial

# Break-even
break_even_vol = CUSTO_FIXO_MENSAL / (log_fato["valor_frete"].mean() - CUSTO_VAR_POR_PED)
break_even_vol = max(0, break_even_vol)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Analise 1 - Estrutura de Custos: Terceirizado vs Internalizado", fontsize=13, fontweight="bold")

# Decomposicao custo fixo
axes[0].pie(list(componentes.values()), labels=list(componentes.keys()),
            autopct="%1.0f%%", startangle=90,
            colors=[COR_RECEITA, COR_FRETE, COR_DESTAQUE, COR_NEUTRO])
axes[0].set_title(f"Composicao do Custo Fixo Mensal\nTotal: R$ {CUSTO_FIXO_MENSAL:,.0f}/mes", fontsize=11)

# Curva de custo
axes[1].plot(volumes, custo_terceirizado/1000, color=COR_FRETE,   linewidth=2, label="Terceirizado (variavel puro)")
axes[1].plot(volumes, custo_proprio/1000,      color=COR_RECEITA, linewidth=2, label="Internalizado total")
axes[1].plot(volumes, custo_hibrido/1000,      color=COR_DESTAQUE,linewidth=2, linestyle="--", label="Hibrido (rotas criticas)")
if 0 < break_even_vol < volumes[-1]:
    axes[1].axvline(break_even_vol, color="black", linewidth=1.5, linestyle=":", alpha=0.7)
    axes[1].annotate(f"Break-even:\n~{break_even_vol:,.0f} ped/mes",
                     xy=(break_even_vol, CUSTO_FIXO_MENSAL/1000),
                     xytext=(break_even_vol*1.1, CUSTO_FIXO_MENSAL/1000*1.3),
                     fontsize=8, arrowprops=dict(arrowstyle="->", lw=1))
axes[1].axvline(n_pedidos_mensal, color=COR_MARGEM, linewidth=1.5, linestyle="--",
                label=f"Volume atual: {n_pedidos_mensal:,.0f}")
axes[1].set_xlabel("Volume mensal de pedidos")
axes[1].set_ylabel("Custo total mensal (R$ mil)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"R$ {v:,.0f}K"))
axes[1].set_title("Curva de Custo por Volume — Make vs Buy", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout()
salvar(fig, "07_estrutura_custos_make_vs_buy")
plt.show()

custo_prop_atual = CUSTO_FIXO_MENSAL + CUSTO_VAR_POR_PED * n_pedidos_mensal
custo_terc_atual = log_fato["valor_frete"].mean() * n_pedidos_mensal
posicao = "JA ULTRAPASSAMOS — internalizacao economicamente superior" if n_pedidos_mensal > break_even_vol else           f"AINDA NAO — faltam {break_even_vol - n_pedidos_mensal:,.0f} ped/mes para o break-even"
print(f"Break-even de volume    : {break_even_vol:,.0f} pedidos/mes")
print(f"Volume atual (media)    : {n_pedidos_mensal:,.0f} pedidos/mes")
print(f"Posicao relativa        : {posicao}")
print(f"Custo terceirizado atual: R$ {custo_terc_atual:,.0f}/mes")
print(f"Custo proprio atual     : R$ {custo_prop_atual:,.0f}/mes")
print(f"Delta mensal            : R$ {custo_terc_atual - custo_prop_atual:+,.0f}/mes")


---

## Análise 2 — Existe um volume a partir do qual a internalização se paga — e já estamos nele?

> *"Internalizar logistica nao e so uma decisao operacional — e uma decisao de alocacao de capital. O dinheiro imobilizado em frota e infraestrutura tem um custo de oportunidade. Calculo o capital necessario, o prazo de imobilizacao e o impacto no fluxo de caixa da empresa adquirida."*

**Framework:** Custo de Capital + Analise de Payback
**Entrega:** Modelagem de capital imobilizado, payback e fluxo de caixa por cenario

**Como este script responde à pergunta:**
> O script projeta o fluxo de caixa acumulado da internalização ao longo de 24 meses. Nos primeiros meses, o fluxo é negativo — o investimento inicial e os custos fixos pesam antes de o volume e a eficiência compensarem. O ponto onde o fluxo acumulado cruza zero é o payback.
>
> 1. **Fluxo de caixa acumulado:** Curva que começa negativa (investimento) e sobe conforme o ganho mensal se acumula. O cruzamento com o eixo zero é o payback. Quanto mais cedo o cruzamento, melhor o retorno.
> 2. **Comparação de cenários:** Mostra o payback nos três cenários (internalizar total, híbrido, manter terceirizado) lado a lado para facilitar a comparação executiva.

**Análise do Resultado:**
Este é o cálculo do Ponto de Equilíbrio (Break-even). Identificamos o volume exato de pedidos onde a economia no frete unitário empata com o novo custo fixo. Saber se já estamos nesse volume determina se a internalização é um projeto de "lucro imediato" ou um investimento para o futuro crescimento da empresa.

In [ ]:
# Ganho mensal esperado com internalização
ganho_frete_cliente = log_fato["valor_frete"].mean() * REDUCAO_FRETE_CLI * n_pedidos_mensal
ganho_volume        = n_pedidos_mensal * AUMENTO_VOLUME * log_fato["preco"].mean() * 0.05  # margem 5%
ganho_total_mensal  = ganho_frete_cliente + ganho_volume
custo_mensal_proprio= CUSTO_FIXO_MENSAL + CUSTO_VAR_POR_PED * n_pedidos_mensal
custo_mensal_terc   = log_fato["valor_frete"].mean() * n_pedidos_mensal
ganho_liq_mensal    = ganho_total_mensal - (custo_mensal_proprio - custo_mensal_terc)

# Fluxo de caixa acumulado — 24 meses
meses = np.arange(1, 25)
# Periodo de maturacao: ramp-up linear
ramp  = np.minimum(1.0, meses / MESES_MATURACAO)
fluxo_mensal  = ganho_liq_mensal * ramp
fluxo_acum    = np.cumsum(fluxo_mensal) - CUSTO_IMPLANTACAO

# Hibrido: 50% do investimento, 60% do ganho
fluxo_hibrido = np.cumsum(ganho_liq_mensal * 0.60 * ramp) - CUSTO_IMPLANTACAO * 0.50

# Payback
payback_meses = next((m for m, v in zip(meses, fluxo_acum) if v >= 0), None)
payback_hibrido = next((m for m, v in zip(meses, fluxo_hibrido) if v >= 0), None)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Analise 2 - Payback e Fluxo de Caixa da Internalizacao", fontsize=13, fontweight="bold")

# Fluxo acumulado
axes[0].plot(meses, fluxo_acum/1000,    color=COR_RECEITA, linewidth=2, label="Internalizacao total")
axes[0].plot(meses, fluxo_hibrido/1000, color=COR_DESTAQUE,linewidth=2, linestyle="--", label="Modelo hibrido")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].fill_between(meses, fluxo_acum/1000, 0, where=fluxo_acum<0, alpha=0.15, color=COR_FRETE)
axes[0].fill_between(meses, fluxo_acum/1000, 0, where=fluxo_acum>=0, alpha=0.15, color=COR_MARGEM)
if payback_meses:
    axes[0].axvline(payback_meses, color=COR_MARGEM, linewidth=1.5, linestyle=":",
                    label=f"Payback total: {payback_meses}m")
if payback_hibrido:
    axes[0].axvline(payback_hibrido, color=COR_DESTAQUE, linewidth=1.5, linestyle=":",
                    label=f"Payback hibrido: {payback_hibrido}m")
axes[0].set_xlabel("Meses apos implantacao")
axes[0].set_ylabel("Fluxo de caixa acumulado (R$ mil)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"R$ {v:,.0f}K"))
axes[0].set_title("Fluxo de Caixa Acumulado por Cenario", fontsize=11)
axes[0].legend(frameon=False, fontsize=8)

# Comparacao de cenarios
cenarios = ["Terceirizado\n(sem mudanca)", "Hibrido\n(rotas criticas)", "Internalizado\ntotal"]
ganhos_24m = [0, float(fluxo_hibrido[-1]), float(fluxo_acum[-1])]
investimentos = [0, CUSTO_IMPLANTACAO * 0.5, CUSTO_IMPLANTACAO]
cores_c = [COR_NEUTRO, COR_DESTAQUE, COR_RECEITA]
bars = axes[1].bar(cenarios, [g/1000 for g in ganhos_24m], color=cores_c, alpha=0.85)
for bar, val, inv in zip(bars, ganhos_24m, investimentos):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                 f"R$ {val/1000:,.0f}K\n(inv: R$ {inv/1000:,.0f}K)",
                 ha="center", fontsize=8)
axes[1].set_ylabel("Ganho liquido acumulado 24m (R$ mil)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"R$ {v:,.0f}K"))
axes[1].set_title("Ganho Liquido Acumulado em 24 Meses por Cenario", fontsize=11)
plt.tight_layout()
salvar(fig, "08_payback_fluxo_caixa")
plt.show()

print(f"Ganho mensal esperado   : R$ {ganho_liq_mensal:,.0f}/mes")
print(f"Capital de implantacao  : R$ {CUSTO_IMPLANTACAO:,.0f}")
print(f"Payback total           : {payback_meses if payback_meses else 'acima de 24 meses'}")
print(f"Payback hibrido         : {payback_hibrido if payback_hibrido else 'acima de 24 meses'}")
print(f"Ganho liq. 24m (total)  : R$ {fluxo_acum[-1]:,.0f}")
print(f"Ganho liq. 24m (hibrido): R$ {fluxo_hibrido[-1]:,.0f}")
print(f"Custo de reversao estim.: R$ {CUSTO_REVERSAO_EST:,.0f}")


---

## Análise 3 — Qual é o custo de capital e o payback do investimento?

> *"A decisão raramente é binária. A solução mais eficiente frequentemente é modular — internalizar onde há vantagem clara e manter terceirizado onde a flexibilidade vale mais. O modelo híbrido focado nas rotas de maior volume (SP, RJ, MG) é simulado para calcular quanto do benefício total é capturado com qual fração do investimento total."*

**Framework:** Análise de portfólio — Make vs Buy seletivo
**Entrega:** Simulação do modelo híbrido com proporção custo/benefício

**Como este script responde à pergunta:**
> O script identifica as rotas que seriam priorizadas no modelo híbrido (alto volume + maior ineficiência logística atual), calcula o percentual de receita e de frete cobrado ao cliente que elas representam, e simula o impacto de internalizar apenas essas rotas. O resultado é uma curva de eficiência: quanto do benefício total captura-se com quanto do investimento total.
>
> 1. **Rotas priorizadas no modelo híbrido:** Lista as rotas que entram no modelo híbrido com o percentual de receita e de frete que representam. O gráfico mostra a relação entre receita coberta e investimento necessário.
> 2. **Curva de eficiência (benefício × investimento):** Para cada conjunto de rotas adicionadas ao modelo híbrido, mostra o percentual do benefício total capturado versus o percentual do investimento total necessário. O ponto de maior inclinação é a fronteira de eficiência ótima.

**Análise do Resultado:**
Aqui medimos a eficiência do capital investido. O payback responde em quantos meses o benefício gerado pela nova logística devolve o dinheiro gasto na sua implementação. Para o investidor, este é o indicador de risco: quanto mais rápido o payback, mais segura é a transição para o novo modelo.

In [ ]:
# Rotas priorizadas: maiores por receita (proxy para modelo hibrido SP/RJ/MG)
top_estados = ["SP", "RJ", "MG", "PR", "RS"]
rotas_hibrido = log_rota[
    log_rota["estado_cliente"].isin(top_estados) |
    log_rota["estado_vendedor"].isin(top_estados)
].sort_values("receita_total", ascending=False)

pct_rec_hibrido   = rotas_hibrido["pct_receita"].sum()
pct_frete_hibrido = rotas_hibrido["frete_total"].sum() / log_rota["frete_total"].sum() * 100
pct_inv_hibrido   = 0.50  # hibrido usa ~50% do investimento total

# Curva de eficiencia: adiciona rotas por receita e mede cobertura
log_rota_s = log_rota.sort_values("receita_total", ascending=False).reset_index(drop=True)
log_rota_s["pct_rec_acum"]   = log_rota_s["pct_receita"].cumsum()
log_rota_s["pct_frete_acum"] = (log_rota_s["frete_total"].cumsum() /
                                  log_rota_s["frete_total"].sum() * 100)
log_rota_s["pct_inv_aprox"]  = np.linspace(5, 100, len(log_rota_s))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Analise 3 - Modelo Hibrido: Beneficio vs Investimento", fontsize=13, fontweight="bold")

# Barras: top rotas por receita
top15r = log_rota.head(15).sort_values("receita_total", ascending=True)
cores_h = [COR_RECEITA if (r in top_estados or any(e in r for e in top_estados))
           else COR_NEUTRO for r in top15r["estado_cliente"]]
axes[0].barh([r[:22] for r in top15r["rota"]], top15r["pct_receita"], color=cores_h, alpha=0.85)
for i, (_, row) in enumerate(top15r.iterrows()):
    axes[0].text(row["pct_receita"]+0.1, i, f"{row['pct_receita']:.1f}%", va="center", fontsize=7)
axes[0].set_xlabel("% da Receita Total")
axes[0].set_title(f"Top 15 Rotas por Receita\n(azul = candidatas ao modelo hibrido)", fontsize=11)
axes[0].legend(handles=[
    mpatches.Patch(color=COR_RECEITA, label=f"Hibrido: {pct_rec_hibrido:.1f}% receita"),
    mpatches.Patch(color=COR_NEUTRO,  label="Terceirizado mantido"),
], frameon=False, fontsize=8)

# Curva de eficiencia
axes[1].plot(log_rota_s["pct_inv_aprox"], log_rota_s["pct_rec_acum"],
             color=COR_RECEITA, linewidth=2, label="% Receita coberta")
axes[1].plot(log_rota_s["pct_inv_aprox"], log_rota_s["pct_frete_acum"],
             color=COR_FRETE, linewidth=2, linestyle="--", label="% Frete ao cliente coberto")
axes[1].axvline(pct_inv_hibrido*100, color=COR_DESTAQUE, linewidth=1.5, linestyle=":",
                label=f"Hibrido: {pct_inv_hibrido*100:.0f}% do investimento")
axes[1].axhline(pct_rec_hibrido, color=COR_NEUTRO, linewidth=1, linestyle="--", alpha=0.5)
axes[1].set_xlabel("% do Investimento Total Necessario")
axes[1].set_ylabel("% do Beneficio Total Capturado")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("Curva de Eficiencia: Beneficio vs Investimento", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout()
salvar(fig, "09_modelo_hibrido_eficiencia")
plt.show()

print(f"Modelo hibrido — rotas priorizadas:")
print(f"  Cobertura de receita    : {pct_rec_hibrido:.1f}%")
print(f"  Cobertura de frete      : {pct_frete_hibrido:.1f}%")
print(f"  Investimento necessario : ~{pct_inv_hibrido*100:.0f}% do total")
print(f"  Eficiencia              : {pct_rec_hibrido/pct_inv_hibrido/100:.1f}x (beneficio/investimento)")


---

## Análise 4 — Um modelo híbrido captura quanto do benefício com qual fração do investimento?

> *"A pergunta não é internalizar ou não internalizar — é internalizar o quê primeiro. Um modelo híbrido bem desenhado captura 70% do benefício com 40% do investimento. O segredo está em priorizar as rotas onde o frete ao cliente é mais alto e o volume justifica o custo fixo marginal de cobertura."*

**Framework:** Análise de eficiência marginal de investimento (curva de Pareto de rotas)  
**Entrega:** Mapa de priorização de rotas para internalização parcial + curva benefício × investimento

**Como este script responde à pergunta:**
> O modelo híbrido é a resposta para quando a internalização total é prematura mas a inação tem custo crescente. Este script constrói a lógica de priorização em duas etapas:
>
> 1. **Ranking de rotas por ROI de internalização:** Ordena cada rota pelo ganho esperado de internalizar dividido pelo custo marginal de cobertura. Rotas com alto frete ao cliente, alto volume e boa densidade geográfica têm ROI maior — são as candidatas naturais à primeira fase do híbrido. O gráfico de barras empilhadas mostra para cada rota priorizada: o ganho de frete ao cliente, o custo fixo marginal e o resultado líquido esperado.
> 2. **Curva de eficiência benefício × investimento:** Plota, à medida que se adicionam rotas ao modelo híbrido, quanto % do benefício total se captura vs quanto % do investimento total se compromete. O ponto de inflexão da curva — onde ela começa a achatar — define o tamanho ótimo do modelo híbrido. Qualquer expansão além desse ponto tem retorno marginal decrescente e pode indicar que a internalização total passa a ser mais eficiente.

**Análise do Resultado:**
Esta análise testa a lei da eficiência: muitas vezes, internalizar apenas as rotas de alto volume (Modelo Híbrido) entrega a maior parte da economia financeira sem exigir o investimento massivo de uma operação 100% própria. É a busca pelo melhor retorno sobre o capital investido com a menor complexidade possível.

In [ ]:
# ─── Análise 4 — Priorização de Rotas para Modelo Híbrido ────────────────────

# Custo marginal por rota: proporcional ao volume relativo
vol_total_rotas = log_rota["n_pedidos"].sum()
log_rota_h = log_rota.copy()
log_rota_h["pct_vol"]         = log_rota_h["n_pedidos"] / vol_total_rotas
log_rota_h["custo_marginal"]  = log_rota_h["pct_vol"] * CUSTO_FIXO_MENSAL
log_rota_h["ganho_frete_cli"] = log_rota_h["frete_medio"] * REDUCAO_FRETE_CLI * log_rota_h["n_pedidos"]
log_rota_h["ganho_volume"]    = log_rota_h["n_pedidos"] * AUMENTO_VOLUME * log_rota_h["frete_medio"] * (1 - REDUCAO_FRETE_CLI)
log_rota_h["ganho_total"]     = log_rota_h["ganho_frete_cli"] + log_rota_h["ganho_volume"]
log_rota_h["resultado_liq"]   = log_rota_h["ganho_total"] - log_rota_h["custo_marginal"]
log_rota_h["roi_rota"]        = log_rota_h["resultado_liq"] / log_rota_h["custo_marginal"].clip(lower=1)

# Ordenar por ROI e calcular curva acumulada
log_rota_h = log_rota_h.sort_values("roi_rota", ascending=False).reset_index(drop=True)
ganho_total_possivel = log_rota_h["ganho_total"].sum()
inv_total_possivel   = log_rota_h["custo_marginal"].sum()
log_rota_h["ganho_acum_pct"] = log_rota_h["ganho_total"].cumsum() / ganho_total_possivel * 100
log_rota_h["inv_acum_pct"]   = log_rota_h["custo_marginal"].cumsum() / inv_total_possivel * 100

# Ponto de inflexão: onde ganho marginal < investimento marginal
log_rota_h["eficiencia_marginal"] = log_rota_h["ganho_total"] / log_rota_h["custo_marginal"].clip(lower=1)
ponto_inflexao_idx = (log_rota_h["eficiencia_marginal"] < 1).idxmax()
if ponto_inflexao_idx == 0:
    ponto_inflexao_idx = len(log_rota_h) // 2
ponto_inf_ganho = log_rota_h.loc[ponto_inflexao_idx, "ganho_acum_pct"]
ponto_inf_inv   = log_rota_h.loc[ponto_inflexao_idx, "inv_acum_pct"]

# Top 10 rotas para gráfico de barras
top10 = log_rota_h.head(10).copy()
labels_rotas = top10["estado_cliente"].astype(str) if "estado_cliente" in top10.columns else top10.index.astype(str)

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Análise 4 — Priorização de Rotas para Modelo Híbrido", fontsize=13, fontweight="bold")

# Barras empilhadas — top 10 rotas
x_r = range(len(top10))
bars_ganho = axes[0].bar(x_r, top10["ganho_total"] / 1000, color=COR_MARGEM, alpha=0.85,
                         label="Ganho total estimado", width=0.55)
bars_custo = axes[0].bar(x_r, -top10["custo_marginal"] / 1000, color=COR_FRETE, alpha=0.75,
                         label="Custo marginal fixo", width=0.55)
axes[0].plot(x_r, top10["resultado_liq"] / 1000, color=COR_DESTAQUE, linewidth=2,
             marker="o", markersize=5, zorder=5, label="Resultado líquido")
axes[0].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_xticks(list(x_r))
axes[0].set_xticklabels(labels_rotas.tolist(), rotation=45, ha="right", fontsize=9)
axes[0].set_ylabel("R$ mil / mês")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"R${v:,.0f}K"))
axes[0].set_title("Top 10 Rotas por ROI de Internalização\n(ordenadas por retorno/investimento)", fontsize=11)
axes[0].legend(frameon=False, fontsize=9)
for spine in ["top", "right"]:
    axes[0].spines[spine].set_visible(False)

# Curva benefício × investimento
axes[1].plot(log_rota_h["inv_acum_pct"], log_rota_h["ganho_acum_pct"],
             color=COR_RECEITA, linewidth=2, label="Curva real")
axes[1].plot([0, 100], [0, 100], color=COR_NEUTRO, linestyle="--", linewidth=1,
             label="Eficiência proporcional (1:1)")
axes[1].scatter([ponto_inf_inv], [ponto_inf_ganho], color=COR_DESTAQUE, s=80, zorder=6,
                label=f"Ponto de inflexão\n({ponto_inf_inv:.0f}% inv. → {ponto_inf_ganho:.0f}% benefício)")
axes[1].fill_between(log_rota_h["inv_acum_pct"], log_rota_h["ganho_acum_pct"],
                     log_rota_h["inv_acum_pct"], alpha=0.07, color=COR_RECEITA)
axes[1].set_xlabel("% do Investimento Total Comprometido")
axes[1].set_ylabel("% do Benefício Total Capturado")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("Curva de Eficiência: Benefício × Investimento\n(ponto de inflexão = tamanho ótimo do híbrido)", fontsize=11)
axes[1].legend(frameon=False, fontsize=9)
for spine in ["top", "right"]:
    axes[1].spines[spine].set_visible(False)

plt.tight_layout()
salvar(fig, "10_priorizacao_hibrido")
plt.show()

n_rotas_otimo = ponto_inflexao_idx + 1
print("\n" + "="*55)
print("INSIGHT — PRIORIZAÇÃO DO MODELO HÍBRIDO")
print("="*55)
print(f"Total de rotas analisadas        : {len(log_rota_h)}")
print(f"Rotas no modelo híbrido ótimo    : {n_rotas_otimo}")
print(f"  Benefício capturado            : {ponto_inf_ganho:.1f}% do total possível")
print(f"  Investimento comprometido      : {ponto_inf_inv:.1f}% do total")
print(f"  Eficiência                     : {ponto_inf_ganho/max(ponto_inf_inv,1):.1f}x")
print(f"Resultado líquido top rota       : R$ {top10['resultado_liq'].iloc[0]:,.0f}/mês")


---

## Análise 5 — Se a internalização não for viável agora, em quanto tempo o crescimento orgânico atingiria o break-even?

> *"Uma decisão de internalização prematura pode destruir capital. Mas esperar também tem custo — cada mês no modelo terceirizado é um mês de frete crescente, SLA deteriorado e conversão pressionada. A análise de horizonte temporal transforma essa tensão em um número: quantos meses de crescimento orgânico são necessários para que a internalização total se torne inevitavelmente superior?"*

**Framework:** Análise de horizonte temporal com sensibilidade a cenários de crescimento  
**Entrega:** Projeção de volume até o break-even em 3 cenários + custo acumulado de espera

**Como este script responde à pergunta:**
> Quando o volume atual está abaixo do break-even, a pergunta não é "se" internalizar — é "quando". Este script responde com precisão:
>
> 1. **Projeção de volume em 3 cenários:** Usando a taxa de crescimento histórica como base, projeta o volume mensal de pedidos em três trajetórias — crescimento pessimista (metade da taxa histórica), base (taxa histórica) e otimista (1,5x a taxa histórica). Para cada cenário, calcula em quantos meses o volume cruzaria o break-even de internalização. O gráfico mostra as três curvas de projeção e a linha de break-even como referência.
> 2. **Custo acumulado de espera:** Para cada mês de espera antes de internalizar, calcula o custo de oportunidade — a diferença entre o que a operação própria teria gerado e o que o modelo terceirizado está custando. A área sob essa curva é o custo total de adiar a decisão. Esse número é o argumento financeiro mais direto para o board: quanto se perde por mês ao não decidir.

**Análise do Resultado:**
Esta é uma visão de longo prazo. Se os números atuais não justificam a mudança imediata, projetamos o crescimento das vendas para saber quando a conta fechará. Este horizonte temporal permite ao comprador decidir se vale a pena "esperar" o negócio amadurecer ou se o tempo de espera é longo demais frente ao risco de mercado.

In [ ]:
# ─── Análise 5 — Horizonte Temporal até o Break-Even ─────────────────────────

# Taxa de crescimento histórica de volume
vol_por_trim = (
    log_fato.groupby("periodo")["id_pedido"]
    .nunique()
    .reset_index(name="n_pedidos")
    .sort_values("periodo")
)
if len(vol_por_trim) >= 2:
    from scipy import stats as _stats
    x_g = np.arange(len(vol_por_trim))
    slope_g, _, _, _, _ = _stats.linregress(x_g, vol_por_trim["n_pedidos"].values)
    cresc_trim_hist = slope_g / vol_por_trim["n_pedidos"].mean()
    cresc_mensal_hist = cresc_trim_hist / 3
else:
    cresc_mensal_hist = 0.02  # fallback 2%/mês

# Cenários de crescimento
cenarios = {
    "Pessimista (0.5x hist.)": cresc_mensal_hist * 0.5,
    "Base (hist.)": cresc_mensal_hist,
    "Otimista (1.5x hist.)": cresc_mensal_hist * 1.5,
}
cores_cen = {"Pessimista (0.5x hist.)": COR_ALERTA,
             "Base (hist.)": COR_RECEITA,
             "Otimista (1.5x hist.)": COR_MARGEM}

MESES_PROJ = 36
resultados_cen = {}
for nome, taxa in cenarios.items():
    vols = [n_pedidos_mensal * (1 + taxa) ** m for m in range(MESES_PROJ + 1)]
    meses_ate_be = next((m for m, v in enumerate(vols) if v >= break_even_vol), None)
    resultados_cen[nome] = {"vols": vols, "meses_ate_be": meses_ate_be, "taxa": taxa}

# Custo acumulado de espera por mês
# Ganho mensal que a operação própria geraria vs custo atual do modelo terceirizado
custo_espera_mensal = ganho_liq_mensal  # ganho líquido que se perde por mês de inação
meses_horizonte = min(filter(None, [r["meses_ate_be"] for r in resultados_cen.values()]),
                      default=MESES_PROJ)
custo_acum = [custo_espera_mensal * m for m in range(MESES_PROJ + 1)]

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Análise 5 — Horizonte Temporal até o Break-Even de Internalização", fontsize=13, fontweight="bold")

# Projeção de volume
meses_x = list(range(MESES_PROJ + 1))
for nome, res in resultados_cen.items():
    axes[0].plot(meses_x, [v / 1000 for v in res["vols"]], color=cores_cen[nome],
                 linewidth=2, label=nome, linestyle="--" if "Pessimista" in nome else "-")
    if res["meses_ate_be"]:
        axes[0].axvline(res["meses_ate_be"], color=cores_cen[nome], linestyle=":",
                        linewidth=1.2, alpha=0.7)
        axes[0].annotate(f"{res['meses_ate_be']}m",
                         xy=(res["meses_ate_be"], break_even_vol / 1000),
                         xytext=(res["meses_ate_be"] + 0.5, break_even_vol / 1000 * 1.05),
                         fontsize=8, color=cores_cen[nome])

axes[0].axhline(break_even_vol / 1000, color="black", linewidth=1.5, linestyle="--",
                label=f"Break-even: {break_even_vol:,.0f} ped/mês")
axes[0].axhline(n_pedidos_mensal / 1000, color=COR_ROXO, linewidth=1, linestyle=":",
                label=f"Volume atual: {n_pedidos_mensal:,.0f} ped/mês")
axes[0].set_xlabel("Meses a partir de hoje")
axes[0].set_ylabel("Volume mensal de pedidos (mil)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1f}K"))
axes[0].set_title("Projeção de Volume até o Break-Even\n(3 cenários de crescimento orgânico)", fontsize=11)
axes[0].legend(frameon=False, fontsize=9)
for spine in ["top", "right"]:
    axes[0].spines[spine].set_visible(False)

# Custo acumulado de espera
axes[1].plot(meses_x, [c / 1000 for c in custo_acum], color=COR_FRETE, linewidth=2)
axes[1].fill_between(meses_x, [c / 1000 for c in custo_acum], alpha=0.12, color=COR_FRETE)
for nome, res in resultados_cen.items():
    if res["meses_ate_be"]:
        custo_no_be = custo_acum[min(res["meses_ate_be"], MESES_PROJ)] / 1000
        axes[1].scatter([res["meses_ate_be"]], [custo_no_be], color=cores_cen[nome],
                        s=60, zorder=6)
        axes[1].annotate(f"{nome.split(' ')[0]}:\nR${custo_no_be:.0f}K perdidos",
                         xy=(res["meses_ate_be"], custo_no_be),
                         xytext=(res["meses_ate_be"] + 1, custo_no_be * 0.9),
                         fontsize=8, color=cores_cen[nome])
axes[1].set_xlabel("Meses de espera antes de internalizar")
axes[1].set_ylabel("Custo acumulado de inação (R$ mil)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"R${v:,.0f}K"))
axes[1].set_title("Custo Acumulado de Espera\n(ganho mensal perdido por mês de inação)", fontsize=11)
for spine in ["top", "right"]:
    axes[1].spines[spine].set_visible(False)

plt.tight_layout()
salvar(fig, "11_horizonte_break_even")
plt.show()

print("\n" + "="*55)
print("INSIGHT — HORIZONTE TEMPORAL")
print("="*55)
print(f"Volume atual           : {n_pedidos_mensal:,.0f} ped/mês")
print(f"Break-even             : {break_even_vol:,.0f} ped/mês")
print(f"Gap atual              : {break_even_vol - n_pedidos_mensal:,.0f} ped/mês")
print(f"Crescimento hist./mês  : {cresc_mensal_hist*100:.1f}%")
for nome, res in resultados_cen.items():
    be_str = f"{res['meses_ate_be']} meses" if res["meses_ate_be"] else ">36 meses"
    print(f"  {nome:<30}: break-even em {be_str}")
print(f"\nCusto de espera/mês    : R$ {custo_espera_mensal:,.0f}")
print(f"Custo de 12 meses de inação: R$ {custo_espera_mensal*12:,.0f}")


---
## Síntese do Bloco 2 — Viabilidade Econômica

> **Limitações desta análise:** os custos fixos e variáveis da operação internalizada são estimativas baseadas em benchmarks de mercado brasileiro — não cotações reais de fornecedores. O ganho de volume com redução de frete assume elasticidade de demanda positiva, hipótese não modelada empiricamente neste bloco. O payback calculado não considera impostos, encargos trabalhistas ou custos de transição. O custo de reversão é uma estimativa conservadora — o valor real depende de contratos firmados e condições de mercado no momento de eventual desfazimento.


In [ ]:
# ─── Veredicto dinamico ───────────────────────────────────────────────────────
_be_vol    = break_even_vol
_vol_atual = n_pedidos_mensal
_acima_be  = _vol_atual > _be_vol
_payback_ok= payback_meses is not None and payback_meses <= 18

# Horizonte temporal (análise 5)
_cresc_m = cresc_mensal_hist
_be_base = resultados_cen["Base (hist.)"]["meses_ate_be"]
_custo_12m = custo_espera_mensal * 12

# Priorização híbrida (análise 4)
_n_rotas_otimo_v = n_rotas_otimo
_eficiencia_v    = ponto_inf_ganho / max(ponto_inf_inv, 1)

_hibrido_ef= pct_rec_hibrido / (pct_inv_hibrido * 100)  # beneficio por % investimento

s_break  = "VOLUME ATUAL ACIMA DO BREAK-EVEN — internalizacao economicamente superior" if _acima_be else f"VOLUME ATUAL ABAIXO DO BREAK-EVEN — faltam {_be_vol-_vol_atual:,.0f} ped/mes"
s_payback= f"PAYBACK EM {payback_meses} MESES" if payback_meses else "PAYBACK ACIMA DE 24 MESES"
s_hibrido= f"EFICIENCIA {_hibrido_ef:.1f}x — hibrido captura {pct_rec_hibrido:.1f}% com {pct_inv_hibrido*100:.0f}% do investimento"

n_alertas = sum([not _acima_be, not _payback_ok])
if n_alertas == 0:
    sinal = "INTERNALIZACAO VIAVEL — break-even superado e payback dentro de 18 meses"
elif n_alertas == 1:
    sinal = "INTERNALIZACAO CONDICIONAL — uma dimensao abaixo do ideal"
else:
    sinal = "MODELO HIBRIDO RECOMENDADO — internalizacao total prematura no volume atual"

print("=" * 65)
print("SINTESE - BLOCO 2: VIABILIDADE ECONOMICA")
print("=" * 65)
print("\n[ BREAK-EVEN ]")
print(f"  Break-even de volume    : {_be_vol:,.0f} pedidos/mes")
print(f"  Volume atual            : {_vol_atual:,.0f} pedidos/mes")
print(f"  Status                  : {s_break}")
print("\n[ PAYBACK ]")
print(f"  Capital de implantacao  : R$ {CUSTO_IMPLANTACAO:,.0f}")
print(f"  Ganho mensal estimado   : R$ {ganho_liq_mensal:,.0f}")
print(f"  Payback total           : {s_payback}")
print(f"  Payback hibrido         : {payback_hibrido if payback_hibrido else 'acima de 24m'} meses")
print("\n[ MODELO HIBRIDO ]")
print(f"  {s_hibrido}")
print("\n[ CUSTO DE REVERSAO ]")
print(f"  Se internalizacao falhar: R$ {CUSTO_REVERSAO_EST:,.0f} para desfazer")
print("\n" + "=" * 65)
print("VEREDICTO PARCIAL - BLOCO 2")
print("=" * 65)
print(f"\nSinal geral: {sinal}")
print("\nProximo passo: Bloco 3 - impacto no SLA e na satisfacao do cliente.")


---
*Próximo notebook: `03_impacto_sla_satisfacao.ipynb` — A internalização entrega SLA melhor — ou apenas troca um problema por outro?*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.
